# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 5: Demografie

Baut auf den in [`01_Exploration.ipynb`](01_Exploration.ipynb) exportierten `eckwerte_long.csv` auf — kein erneutes Einlesen der Excel-Rohdaten nötig. [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb) zeigt Altersstruktur und Geschlecht bisher nur als **Momentaufnahme des letzten Berichtsmonats**. Dieses Notebook macht daraus Zeitreihen und ergänzt zwei bislang komplett unanalysierte Merkmale, die in den Rohdaten in derselben Zeilengruppe stehen: **Ausländer** und **schwerbehinderte Menschen** unter den Arbeitslosen (Rechtskreis SGB II, Januar 2025 – September 2026).

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROCESSED_DIR = Path("..") / "Data" / "processed"

df_long = pd.read_csv(PROCESSED_DIR / "eckwerte_long.csv")
df_long["period"] = pd.PeriodIndex(df_long["period"], freq="M").to_timestamp()

kennzahlen = pd.read_csv(PROCESSED_DIR / "kennzahlen_sgb2.csv", index_col="period", parse_dates=["period"])
kennzahlen = kennzahlen.sort_index()

merkmale = [
    "15 bis unter 25 Jahre", "25 bis unter 50 Jahre", "50 Jahre und älter", "55 Jahre und älter",
    "Männer", "Frauen", "Ausländer", "schwerbehinderte Menschen",
]
demografie = df_long[(df_long["kategorie"] == "Arbeitslose") & (df_long["merkmal"].isin(merkmale))]
demografie = demografie.pivot_table(index="period", columns="merkmal", values="sgb2").sort_index()
demografie.tail()

merkmal,15 bis unter 25 Jahre,25 bis unter 50 Jahre,50 Jahre und älter,55 Jahre und älter,Ausländer,Frauen,Männer,schwerbehinderte Menschen
period,,,,,,,,
2026-05-01,166.0,888.0,498.0,343.0,711.0,716.0,836.0,136.0
2026-06-01,154.0,855.0,491.0,338.0,681.0,682.0,818.0,130.0
2026-07-01,170.0,848.0,479.0,344.0,663.0,693.0,804.0,136.0
2026-08-01,228.0,863.0,473.0,330.0,711.0,716.0,848.0,140.0
2026-09-01,202.0,835.0,486.0,347.0,685.0,707.0,816.0,134.0


## 1. Altersstruktur im Zeitverlauf

Die drei Altersgruppen **15 bis unter 25**, **25 bis unter 50** und **50 Jahre und älter** sind überschneidungsfrei und summieren sich zum Gesamtbestand — daher als gestapelte Fläche dargestellt. **55 Jahre und älter** ist eine Teilmenge von „50+“ und wird separat als Linie ergänzt.

In [2]:
alter_gruppen = ["15 bis unter 25 Jahre", "25 bis unter 50 Jahre", "50 Jahre und älter"]
demografie["50 bis unter 55 Jahre"] = demografie["50 Jahre und älter"] - demografie["55 Jahre und älter"]

alter_stapel = ["15 bis unter 25 Jahre", "25 bis unter 50 Jahre", "50 bis unter 55 Jahre", "55 Jahre und älter"]
farben_alter = {
    "15 bis unter 25 Jahre": "#2563eb", "25 bis unter 50 Jahre": "#7c3aed",
    "50 bis unter 55 Jahre": "#f59e0b", "55 Jahre und älter": "#dc2626",
}

fig = go.Figure()
for gruppe in alter_stapel:
    fig.add_trace(go.Scatter(
        x=demografie.index, y=demografie[gruppe], name=gruppe,
        mode="lines", stackgroup="alter", line=dict(color=farben_alter[gruppe], width=0.5),
    ))
fig.update_layout(
    title="Altersstruktur der Arbeitslosen im Zeitverlauf (SGB II)",
    yaxis_title="Personen",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=460,
)
fig

Die Altersstruktur ist über den gesamten Zeitraum bemerkenswert stabil: Die Gruppe **25 bis unter 50 Jahre** stellt durchgehend die größte Gruppe (57,7 % im Januar 2025, 54,8 % im September 2026), gefolgt von **50 Jahre und älter** (32,2 % → 31,9 %) und **15 bis unter 25 Jahre** (10,1 % → 13,3 %). Der höhere Jugendanteil am Ende ist ein wiederkehrender Saisoneffekt: Wie schon im August/September 2025 (14,0 % / 13,6 %) steigt er auch im August/September 2026 (14,6 % / 13,3 %) sprunghaft an — typisch für die Zeit nach Schul- und Ausbildungsende. Damit die Teilmenge „55 Jahre und älter“ im Chart als eigenes, korrekt eingeordnetes Band erscheint (nicht als frei schwebende Linie), wird die Gruppe 50+ für die Darstellung in „50 bis unter 55“ und „55 und älter“ aufgeteilt — beide zusammen ergeben wieder die 50+-Gruppe. Auffälliger als die grobe Verteilung ist die Entwicklung *innerhalb* dieser Aufteilung: Der Anteil der 55-Jährigen und Älteren an der Gruppe 50+ steigt von 64,8 % auf 71,4 % (Höchstwert 72,1 % im Januar 2026) — die Alterung innerhalb der älteren Arbeitslosen nimmt also stärker zu, als es die grobe 50+-Gruppe allein zeigt.

In [3]:
alter_anteile = demografie[alter_gruppen].div(demografie[alter_gruppen].sum(axis=1), axis=0) * 100
alter_anteile.round(1)

merkmal,15 bis unter 25 Jahre,25 bis unter 50 Jahre,50 Jahre und älter
period,,,
2025-01-01,10.1,57.7,32.2
2025-02-01,10.0,58.3,31.7
2025-03-01,10.0,59.1,30.9
2025-04-01,11.0,58.6,30.4
2025-05-01,10.2,58.1,31.6
2025-06-01,11.7,56.2,32.1
2025-07-01,11.4,56.3,32.3
2025-08-01,14.0,55.1,30.9
2025-09-01,13.6,55.1,31.3


Die Anteilstabelle bestätigt den optischen Eindruck aus der Fläche: Die prozentuale Verschiebung zwischen den drei Altersgruppen bleibt über 21 Monate im Bereich weniger Prozentpunkte (am stärksten schwankt saisonbedingt die jüngste Gruppe) — deutlich stabiler als z. B. die Langzeitarbeitslosigkeit (siehe [`03_Statistik.ipynb`](03_Statistik.ipynb)).

## 2. Geschlechterverteilung im Zeitverlauf

In [4]:
anteil_frauen = (demografie["Frauen"] / (demografie["Frauen"] + demografie["Männer"]) * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=demografie.index, y=demografie["Männer"], name="Männer",
    mode="lines", stackgroup="geschlecht", line=dict(color="#2563eb", width=0.5),
))
fig.add_trace(go.Scatter(
    x=demografie.index, y=demografie["Frauen"], name="Frauen",
    mode="lines", stackgroup="geschlecht", line=dict(color="#db2777", width=0.5),
))
fig.update_layout(
    title="Geschlechterverteilung der Arbeitslosen im Zeitverlauf (SGB II)",
    yaxis_title="Personen",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=420,
)
fig

Auch die Geschlechterverteilung ist relativ stabil: Der Männeranteil liegt im Januar 2025 bei 53,1 % und im September 2026 bei 53,6 %, der Frauenanteil entsprechend bei 46,9 % bzw. 46,4 % (Spanne über den gesamten Zeitraum: 45,0–47,4 % Frauen). Die im Juni 2026 noch sichtbare Verschiebung zu mehr Männern (54,5 %) hat sich in den Folgemonaten wieder teilweise zurückgebildet. In absoluten Zahlen: Männer 780 → 816 Personen, Frauen 690 → 707 Personen — der Bestandsanstieg seit Anfang 2025 geht damit leicht überproportional auf die Gruppe der Männer zurück.

## 3. Ausländische Arbeitslose im Zeitverlauf

Bisher komplett unanalysiertes Merkmal. Da „Ausländer“ eine Querschnittsgruppe ist (überschneidet sich mit Alter und Geschlecht), wird sie separat betrachtet — als absolute Zahl und als Anteil am Gesamtbestand.

In [5]:
anteil_auslaender = (demografie["Ausländer"] / kennzahlen["Arbeitslose (Bestand)"] * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=demografie.index, y=demografie["Ausländer"], name="Ausländer (absolut)",
    mode="lines+markers", line=dict(color="#0d9488"), marker=dict(size=6), yaxis="y1",
))
fig.add_trace(go.Scatter(
    x=anteil_auslaender.index, y=anteil_auslaender, name="Anteil am Bestand (%)",
    mode="lines+markers", line=dict(color="#f59e0b", dash="dot"), marker=dict(size=6), yaxis="y2",
))
fig.update_layout(
    title="Ausländische Arbeitslose im Zeitverlauf (SGB II)",
    yaxis=dict(title="Personen"),
    yaxis2=dict(title="Anteil (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=420,
)
fig

Ausländische Arbeitslose sind mit einem Anteil zwischen 44,3 % und 51,7 % ein **überraschend großer Teil** des SGB-II-Arbeitslosenbestands — deutlich mehr, als man ohne diese Auswertung vermuten würde. Der Anteil ist über den Zeitraum spürbar gesunken: von 51,7 % (Januar 2025, 760 Personen) auf 45,0 % (September 2026, 685 Personen; Tiefstwert 44,3 % im Juli 2026) — ein Rückgang, der zeitlich mit der insgesamt rückläufigen Entwicklung der Bedarfsgemeinschaften zusammenfällt (siehe [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb), Kapitel „Bedarfsgemeinschaften und Personen in BG“).

## 4. Schwerbehinderte Menschen im Zeitverlauf

Ebenfalls bisher unanalysiert, gleiche Darstellung wie Ausländer.

In [6]:
anteil_schwerbehindert = (demografie["schwerbehinderte Menschen"] / kennzahlen["Arbeitslose (Bestand)"] * 100).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=demografie.index, y=demografie["schwerbehinderte Menschen"], name="Schwerbehinderte (absolut)",
    mode="lines+markers", line=dict(color="#7c3aed"), marker=dict(size=6), yaxis="y1",
))
fig.add_trace(go.Scatter(
    x=anteil_schwerbehindert.index, y=anteil_schwerbehindert, name="Anteil am Bestand (%)",
    mode="lines+markers", line=dict(color="#f59e0b", dash="dot"), marker=dict(size=6), yaxis="y2",
))
fig.update_layout(
    title="Schwerbehinderte Menschen unter den Arbeitslosen im Zeitverlauf (SGB II)",
    yaxis=dict(title="Personen"),
    yaxis2=dict(title="Anteil (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=420,
)
fig

Schwerbehinderte Menschen machen einen kleineren, aber ebenfalls über die Zeit leicht wachsenden Anteil aus: von 7,9 % (116 Personen, Januar 2025) auf 8,8 % (134 Personen, September 2026), mit einer Spanne von 100 bis 144 Personen (7,0–9,2 %) im Beobachtungszeitraum.

## 5. Export der aufbereiteten Daten

Grundlage für den Dashboard-Tab „👥 Demografie“.

In [7]:
export = demografie.copy()
export["Anteil Frauen (%)"] = anteil_frauen
export["Anteil Ausländer (%)"] = anteil_auslaender
export["Anteil Schwerbehinderte (%)"] = anteil_schwerbehindert
export.to_csv(PROCESSED_DIR / "demografie_sgb2.csv")
export.tail()

merkmal,15 bis unter 25 Jahre,25 bis unter 50 Jahre,50 Jahre und älter,55 Jahre und älter,Ausländer,Frauen,Männer,schwerbehinderte Menschen,50 bis unter 55 Jahre,Anteil Frauen (%),Anteil Ausländer (%),Anteil Schwerbehinderte (%)
period,,,,,,,,,,,,
2026-05-01,166.0,888.0,498.0,343.0,711.0,716.0,836.0,136.0,155.0,46.1,45.8,8.8
2026-06-01,154.0,855.0,491.0,338.0,681.0,682.0,818.0,130.0,153.0,45.5,45.4,8.7
2026-07-01,170.0,848.0,479.0,344.0,663.0,693.0,804.0,136.0,135.0,46.3,44.3,9.1
2026-08-01,228.0,863.0,473.0,330.0,711.0,716.0,848.0,140.0,143.0,45.8,45.5,9.0
2026-09-01,202.0,835.0,486.0,347.0,685.0,707.0,816.0,134.0,139.0,46.4,45.0,8.8


## 6. Zusammenfassung

- **Altersstruktur:** weitgehend stabil (25–50 Jahre dominiert mit 55–59 %), mit einem wiederkehrenden Sommereffekt bei den unter 25-Jährigen (August/September jeweils 13–15 %). Innerhalb der Gruppe 50+ altert die Zusammensetzung spürbar: Der Anteil der 55-Jährigen und Älteren steigt von 64,8 % auf 71,4 % der 50+-Gruppe.
- **Geschlecht:** weitgehend stabil (Männeranteil 53,1 % → 53,6 %); die zwischenzeitliche Verschiebung zu mehr Männern (54,5 % im Juni 2026) hat sich teilweise zurückgebildet.
- **Ausländer:** mit 44–52 % ein überraschend großer, deutlich rückläufiger Anteil des Bestands (760 → 685 Personen, 51,7 % → 45,0 %) — die bislang auffälligste neue Erkenntnis dieses Notebooks.
- **Schwerbehinderte Menschen:** kleinerer, leicht steigender Anteil (7,9 % → 8,8 %, 116 → 134 Personen).

Insgesamt zeigt sich: Alter und Geschlecht der SGB-II-Arbeitslosen sind über den Beobachtungszeitraum bemerkenswert stabil, während der Ausländeranteil eine deutliche und bislang unbeachtete Dynamik aufweist, die eine genauere Betrachtung lohnt (z. B. im Zusammenhang mit Zu-/Abwanderung oder Integration in den Arbeitsmarkt — mit den aktuell verfügbaren Kennzahlen nicht weiter aufschlüsselbar).

Weiter geht es mit der Umsetzung im [Streamlit-Dashboard](../Dashboard/app.py) (Tab „👥 Demografie“) sowie optional mit `06_Erwerbstaetigkeit.ipynb` (Aufstocker, ⚠️ 3 Monate Wartezeit).